# Graph Neural Networks on Brain Connectomes

Model the brain as a graph: regions = nodes, functional connections = edges. Train a GNN (Graph Attention Network) to classify subjects. This is at the cutting edge of computational neuroscience + deep learning.

**Frontier context:** GNNs on connectomes are being used to predict psychiatric disorders (schizophrenia, ADHD, ASD) with state-of-the-art accuracy, and to identify subnetworks that drive disease.

## Prerequisites

- **Python 3.8+** with PyTorch and PyTorch Geometric installed
- Understanding of graph theory basics (nodes, edges, adjacency matrices)
- Familiarity with functional connectivity and brain parcellations
- The ADHD200 sample and Schaefer atlas will be downloaded automatically by nilearn

## Setup

Run the cell below to install all required packages. PyTorch Geometric requires compatible PyTorch and CUDA versions -- see https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html

In [ ]:
# Install required packages
# NOTE: torch-geometric may need version-specific install.
# See https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html
!pip install -q torch numpy matplotlib nilearn nibabel scipy
!pip install -q torch-geometric

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader as GeoDataLoader
from torch_geometric.nn import GATConv, global_mean_pool
import numpy as np
import matplotlib.pyplot as plt
import nilearn.datasets as nld
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure
import torch_geometric

print('torch_geometric', torch_geometric.__version__)

In [ ]:
# Build connectomes from ADHD200 dataset (1 subject)
adhd = nld.fetch_adhd(n_subjects=1)
schaefer = nld.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7)

# resampling_target='data' resamples atlas to fMRI space -- fast
# fit() once so parcel count is consistent; transform() each subject
masker = NiftiLabelsMasker(labels_img=schaefer.maps, standardize='zscore_sample',
                            resampling_target='data')
masker.fit()
ts = masker.transform(adhd.func[0])
print(f'Time series: {ts.shape}  (TRs x nodes)')

conn = ConnectivityMeasure(kind='correlation')
fc = conn.fit_transform([ts])[0]
print(f'FC matrix: {fc.shape}')

# Fast parcel centroids via scipy (avoids slow find_parcellation_cut_coords)
import nibabel as nib
from scipy import ndimage
atlas_img = nib.load(schaefer.maps)
atlas_data = atlas_img.get_fdata().astype(int)
affine = atlas_img.affine
region_coords = np.array([
    affine[:3, :3] @ np.array(ndimage.center_of_mass(atlas_data == i)) + affine[:3, 3]
    for i in range(1, 101)
])
print(f'Region coords: {region_coords.shape}')

import nilearn.plotting as nlp
nlp.plot_connectome(fc, region_coords, edge_threshold='85%',
                    title='Brain graph (Schaefer 100 nodes, top 15% edges)')
plt.show()

In [ ]:
# Convert connectome -> PyG graph
def connectome_to_graph(fc_matrix, node_features, threshold=0.3, label=0):
    """Convert FC matrix to PyTorch Geometric Data object."""
    n = fc_matrix.shape[0]
    # Threshold edges
    adj = (np.abs(fc_matrix) > threshold).astype(float)
    np.fill_diagonal(adj, 0)
    edges = np.array(np.where(adj))
    edge_weights = fc_matrix[edges[0], edges[1]]

    x = torch.FloatTensor(node_features)           # node features
    edge_index = torch.LongTensor(edges)            # connectivity
    edge_attr = torch.FloatTensor(edge_weights).unsqueeze(1)  # weights
    y = torch.LongTensor([label])
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

# Use FC row profiles as node features (each node's connectivity pattern)
graph = connectome_to_graph(fc, fc, threshold=0.3, label=0)
print(f'Graph: {graph.num_nodes} nodes, {graph.num_edges} edges')
print(f'Node features: {graph.x.shape}')

In [ ]:
# Graph Attention Network (GAT) -- state-of-the-art for connectome classification
class BrainGAT(torch.nn.Module):
    def __init__(self, in_channels, hidden=64, heads=4, n_classes=2):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden, heads=heads, dropout=0.3)
        self.conv2 = GATConv(hidden * heads, hidden, heads=1, concat=False, dropout=0.3)
        self.classifier = torch.nn.Linear(hidden, n_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        x = global_mean_pool(x, data.batch if data.batch is not None
                              else torch.zeros(data.num_nodes, dtype=torch.long))
        return self.classifier(x)

model = BrainGAT(in_channels=graph.x.shape[1], hidden=64, heads=4, n_classes=2)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Demo forward pass
model.eval()
with torch.no_grad():
    out = model(graph)
    probs = F.softmax(out, dim=1)
print(f'Output probabilities (CTRL / ADHD): {probs.numpy()}')
print('\nFor real classification: load multi-subject ADHD200 data, train with cross-validation.')
print('See: https://doi.org/10.1016/j.neuroimage.2022.119417 for reference GNN on connectomes.')

## References

- Schaefer, A., Kong, R., Gordon, E. M., et al. (2018). Local-global parcellation of the human cerebral cortex from intrinsic functional connectivity MRI. *Cerebral Cortex*, 28(9), 3095-3114. https://doi.org/10.1093/cercor/bhx179
- Velickovic, P., Cucurull, G., Casanova, A., et al. (2018). Graph attention networks. *ICLR*. https://arxiv.org/abs/1710.10903
- Li, X., Zhou, Y., Dvornek, N., et al. (2021). BrainGNN: Interpretable brain graph neural network for fMRI analysis. *Medical Image Analysis*, 74, 102233. https://doi.org/10.1016/j.media.2021.102233
- Fey, M. & Lenssen, J. E. (2019). Fast graph representation learning with PyTorch Geometric. *ICLR Workshop on Representation Learning on Graphs and Manifolds*.
- The ADHD-200 Consortium (2012). The ADHD-200 Consortium: A model to advance the translational potential of neuroimaging in clinical neuroscience. *Frontiers in Systems Neuroscience*, 6, 62.